<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-cartpole-dqn-lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CartPole-v1 - DQN with PyTorch Lightning

This notebook trains a **Deep Q-Network (DQN)** agent on the classic CartPole-v1 environment using **PyTorch Lightning**.

In [1]:
!pip install gymnasium[classic-control] pytorch-lightning wandb tsilva_notebook_utils==0.0.57 > /dev/null

🔑 Loading API keys and authentication tokens from Colab secrets:

In [2]:
from tsilva_notebook_utils.colab import load_secrets_into_env

# TODO: move to tsilva_notebook_utils
def load_secrets_into_env(keys):
    import os
    from dotenv import load_dotenv
    load_dotenv(override=True)

    try:
        from google.colab import userdata
        for key in keys:
            value = userdata.get(key)
            assert value, f"Key {key} not found in userdata"
            os.environ[key] = value
    except:
        from dotenv import load_dotenv
        load_dotenv(override=True)

    values = []
    for key in keys:
        value = os.getenv(key)
        assert value, f"Key {key} not found in environment variables"
        values.append(value)


_ = load_secrets_into_env([
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

Define configuration:

In [3]:
import os
import random, math
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import pytorch_lightning as pl
from collections import deque
from tsilva_notebook_utils.colab import notebook_id_from_title

def setup_config():
    # Set notebook ID in the environment
    #os.environ["NOTEBOOK_ID"] = notebook_id_from_title()

    return {
        'env_id': 'CartPole-v1',
        'hidden_size': 128,
        'gamma': 0.99,
        'batch_size': 64,
        'replay_size': 10000,
        'lr': 1e-3,
        'eps_start': 1.0,
        'eps_end': 0.05,
        'eps_decay': 500,
        'target_update': 10,
        'max_steps': 5000,
        'seed': 42,
    }

CONFIG = setup_config()

pl.seed_everything(CONFIG['seed'], workers=True)

Seed set to 42


42

Login to wandb:

In [4]:
from wandb import login
login()

wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
env = gym.make(CONFIG['env_id'])
state, _ = env.reset(seed=CONFIG['seed'])
print(state.shape, env.action_space.n)

(4,) 2


Create replay buffer:

In [6]:
env = gym.make(CONFIG['env_id'])
N_INPUTS = env.observation_space.shape[0]
N_OUTPUTS = int(env.action_space.n)
N_INPUTS, N_OUTPUTS

(4, 2)

In [7]:
state, _ = env.reset(seed=CONFIG['seed'])
state

array([ 0.0273956 , -0.00611216,  0.03585979,  0.0197368 ], dtype=float32)

Create replay buffer:

In [8]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, *transition):
        self.buffer.append(transition)

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return map(np.stack, zip(*batch))
    
    def __len__(self):
        return len(self.buffer)
    
    def __repr__(self):
        return f"ReplayBuffer(size={self.buffer.maxlen})"
    
replay_buffer = ReplayBuffer(CONFIG['replay_size'])
replay_buffer

ReplayBuffer(size=10000)

In [9]:
replay_buffer = ReplayBuffer(CONFIG['replay_size'])
for _ in range(10):
    replay_buffer.push(np.random.rand(*state.shape), 0, 1.0, np.random.rand(*state.shape), False)
states, actions, rewards, next_states, dones = list(replay_buffer.sample(2))
dict(
    length=len(replay_buffer),
    states=states, 
    actions=actions, 
    rewards=rewards, 
    next_states=next_states, 
    dones=dones
)

{'length': 10,
 'states': array([[0.37454012, 0.95071431, 0.73199394, 0.59865848],
        [0.06505159, 0.94888554, 0.96563203, 0.80839735]]),
 'actions': array([0, 0]),
 'rewards': array([1., 1.]),
 'next_states': array([[0.15601864, 0.15599452, 0.05808361, 0.86617615],
        [0.30461377, 0.09767211, 0.68423303, 0.44015249]]),
 'dones': array([False, False])}

In [10]:
class DQNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_INPUTS, CONFIG['hidden_size']),
            nn.ReLU(),
            nn.Linear(CONFIG['hidden_size'], N_OUTPUTS)
        )

    def forward(self, x):
        return self.net(x)
    
model = DQNModel()
model

DQNModel(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [11]:
model = DQNModel()
state, _ = env.reset(seed=CONFIG['seed'])
with torch.no_grad(): q_values = model(torch.tensor(state).float().unsqueeze(0)) # Model expects a batch dimension, hence the unsqueeze(0)
q_values

tensor([[-0.0821,  0.0446]])

Select the best action based on predicted Q-values:

In [12]:
actions = torch.argmax(q_values, dim=1) # Since q-values are a batch of values, we take the argmax across the action dimension
actions

tensor([1])

In [13]:
def act(model, state, step):
    eps = CONFIG['eps_end'] + (CONFIG['eps_start']-CONFIG['eps_end'])*math.exp(-step/CONFIG['eps_decay'])
    if random.random() < eps: return int(env.action_space.sample())
    with torch.no_grad(): return int(torch.argmax(model(torch.tensor(state).float().unsqueeze(0)), 1))

action = act(model, state, 0)
action

1

In [15]:
states, actions, rewards, next_states, dones = list(replay_buffer.sample(2))
states = torch.tensor(states, dtype=torch.float32)
actions = torch.tensor(actions, dtype=torch.long).unsqueeze(-1)
rewards = torch.tensor(rewards, dtype=torch.float32)
next_states = torch.tensor(next_states, dtype=torch.float32)
dones = torch.tensor(dones, dtype=torch.float32)
q_values = model(states).gather(1, actions).squeeze()
next_q = model(next_states).max(1)[0]
target = rewards + CONFIG['gamma']*next_q*(1-dones)
nn.functional.mse_loss(q_values, target.detach())

tensor(2.1931, grad_fn=<MseLossBackward0>)

In [16]:
states, actions, rewards, next_states, dones = list(replay_buffer.sample(2))
states = torch.tensor(states, dtype=torch.float32)
states.shape, states

(torch.Size([2, 4]),
 tensor([[0.2809, 0.5427, 0.1409, 0.8022],
         [0.6011, 0.7081, 0.0206, 0.9699]]))

In [17]:
states, actions, rewards, next_states, dones = list(replay_buffer.sample(2))
actions = torch.tensor(actions, dtype=torch.long)
actions.shape, actions

(torch.Size([2]), tensor([0, 0]))

Reshape actions to have a batch dimension as well:

In [18]:
states, actions, rewards, next_states, dones = list(replay_buffer.sample(2))
actions = torch.tensor(actions, dtype=torch.long).unsqueeze(-1)  # Unsqueeze to match the shape of q-values
actions.shape, actions

(torch.Size([2, 1]),
 tensor([[0],
         [0]]))

In [19]:
states, actions, rewards, next_states, dones = list(replay_buffer.sample(2))
states = torch.tensor(states, dtype=torch.float32)
actions = torch.tensor(actions, dtype=torch.long).unsqueeze(-1)
with torch.no_grad(): q_values = model(states)
q_values.shape, q_values

(torch.Size([2, 2]),
 tensor([[-0.3054,  0.3054],
         [-0.2370,  0.1826]]))

In [20]:
states, actions, rewards, next_states, dones = list(replay_buffer.sample(2))
states = torch.tensor(states, dtype=torch.float32)
actions = torch.tensor(actions, dtype=torch.long).unsqueeze(-1)
with torch.no_grad(): q_values = model(states).gather(1, actions).squeeze()
q_values.shape, q_values

(torch.Size([2]), tensor([-0.2370, -0.2444]))

In [21]:
states, actions, rewards, next_states, dones = list(replay_buffer.sample(2))
rewards = torch.tensor(rewards, dtype=torch.float32)
dones = torch.tensor(dones, dtype=torch.float32)
rewards.shape, rewards, dones.shape, dones

(torch.Size([2]), tensor([1., 1.]), torch.Size([2]), tensor([0., 0.]))

In [22]:
states, actions, rewards, next_states, dones = list(replay_buffer.sample(2))
next_states = torch.tensor(next_states, dtype=torch.float32)
actions = torch.tensor(actions, dtype=torch.long).unsqueeze(-1)
rewards = torch.tensor(rewards, dtype=torch.float32)
with torch.no_grad(): next_q = model(next_states)
next_q.shape, next_q, next_q.max(1)[0].shape, next_q.max(1)[0]

(torch.Size([2, 2]),
 tensor([[-0.1938, -0.0301],
         [-0.2442,  0.1511]]),
 torch.Size([2]),
 tensor([-0.0301,  0.1511]))

In [23]:
states, actions, rewards, next_states, dones = list(replay_buffer.sample(2))
states = torch.tensor(states, dtype=torch.float32)
actions = torch.tensor(actions, dtype=torch.long).unsqueeze(-1)
rewards = torch.tensor(rewards, dtype=torch.float32)
next_states = torch.tensor(next_states, dtype=torch.float32)
next_states.shape, next_states

(torch.Size([2, 4]),
 tensor([[0.0746, 0.9869, 0.7722, 0.1987],
         [0.9395, 0.8948, 0.5979, 0.9219]]))

In [24]:
states, actions, rewards, next_states, dones = list(replay_buffer.sample(2))
states = torch.tensor(states, dtype=torch.float32)
actions = torch.tensor(actions, dtype=torch.long).unsqueeze(-1)
rewards = torch.tensor(rewards, dtype=torch.float32)
dones = torch.tensor(dones, dtype=torch.float32)
next_states = torch.tensor(next_states, dtype=torch.float32)
with torch.no_grad(): next_q = model(next_states).max(1)[0]
q_targets = rewards + CONFIG['gamma'] * next_q * (1 - dones)
q_values, q_targets

(tensor([-0.2370, -0.2444]), tensor([1.1496, 1.1014]))

In [25]:
loss = nn.functional.mse_loss(q_values, q_targets)
loss

tensor(1.8668)

In [28]:
class _Infinite(torch.utils.data.IterableDataset):
    def __iter__(self):
        while True:
            yield 0
infinite_data_loader = torch.utils.data.DataLoader(_Infinite(), batch_size=1)
infinite_data_loader

class DQNModule(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.save_hyperparameters()

        # Model used to predict Q-values for each action in a given state
        self.q_model = DQNModel()

        # Target network used to stabilize training
        self.target_model = DQNModel()
        self.target_model.load_state_dict(self.q_model.state_dict())

        # Create environment
        self.env = gym.make(CONFIG['env_id'])
        self.state, _ = self.env.reset(seed=CONFIG['seed'])

        # Create replay buffer
        self.buffer = ReplayBuffer(CONFIG['replay_size'])

        self.total_steps = 0
        self.episode_steps = 0
        self.episode_reward = 0
        
    def forward(self, x):
        return self.q_model(x)

    def act(self, state):
        # TODO: extract into util
        eps = CONFIG['eps_end'] + (CONFIG['eps_start'] - CONFIG['eps_end']) * math.exp(-1.0 * self.total_steps / CONFIG['eps_decay'])

        # If random number is less than epsilon, take a random action (exploration)
        if random.random() < eps:
            return self.env.action_space.sample()
        # Otherwise, use the model to predict the best action (exploitation)
        else:
            state = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            with torch.no_grad(): q = self.q_model(state)
            return int(torch.argmax(q, dim=1)[0].item())

    def train_dataloader(self):
        return infinite_data_loader

    # TODO: when is this called?
    def training_step(self, batch, batch_idx):
        action = self.act(self.state)
        next_state, reward, terminated, truncated, _ = self.env.step(action)
        done = terminated or truncated
        self.buffer.push(self.state, action, reward, next_state, done)
        self.state = next_state

        self.total_steps += 1
        self.episode_steps += 1
        self.episode_reward += reward

        # If the buffer still doesn't have enough samples, skip the training step
        loss = None
        if len(self.buffer) >= CONFIG['batch_size']: # TODO: min != batch_size
            # Sample a batch from the replay buffer
            states, actions, rewards, next_states, dones = self.buffer.sample(CONFIG['batch_size'])
            states = torch.tensor(states, dtype=torch.float32, device=self.device)
            actions = torch.tensor(actions, dtype=torch.long, device=self.device).unsqueeze(-1)
            rewards = torch.tensor(rewards, dtype=torch.float32, device=self.device)
            next_states = torch.tensor(next_states, dtype=torch.float32, device=self.device)
            dones = torch.tensor(dones, dtype=torch.float32, device=self.device)

            # Compute the loss
            q_values = self.q_model(states).gather(1, actions).squeeze()
            next_q = self.target_model(next_states).max(1)[0]
            targets = rewards + CONFIG['gamma'] * next_q * (1 - dones)
            loss = nn.functional.mse_loss(q_values, targets.detach())
            self.log('loss', loss, on_step=True, prog_bar=True) # TODO: what is this log?

            # Every N steps, update the target network
            if self.global_step % CONFIG['target_update'] == 0:
                self.target_model.load_state_dict(self.q_model.state_dict())

        if done:
            self.log('episode_reward', self.episode_reward, on_step=True, prog_bar=True)
            self.log('episode_steps', self.episode_steps, on_step=True, prog_bar=True)
            self.episode_steps = 0
            self.episode_reward = 0
            self.state = self.env.reset()[0]

        # TODO: why?
        # Return the current loss
        return loss

    def configure_optimizers(self):
        # TODO: better with other optimizer?
        return torch.optim.Adam(self.q_model.parameters(), lr=CONFIG['lr'])

module = DQNModule()
module

DQNModule(
  (q_model): DQNModel(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=2, bias=True)
    )
  )
  (target_model): DQNModel(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=2, bias=True)
    )
  )
)

In [29]:
import pytorch_lightning as pl

class StopOnReward(pl.Callback):
    """Immediately terminates training once the episode reward ≥ target."""
    def __init__(self, target_reward: float):
        super().__init__()
        self.target_reward = target_reward

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        # `episode_reward` is logged inside `training_step`
        r = trainer.callback_metrics.get("episode_reward")
        if r is not None and r >= self.target_reward:
            print(f"Stopping: reward {r:.1f} ≥ target {self.target_reward}")
            trainer.should_stop = True      # Lightning 2.x
            # trainer.strategy.request_stop()  # Lightning < 2.0

reward_cb = StopOnReward(target_reward=475)   # CartPole-v1 is “solved” at 475+
    
# TODO: stop when treshold is reached
module = DQNModule()
trainer = pl.Trainer(
    log_every_n_steps=50, 
    enable_model_summary=False,
    callbacks=[reward_cb]
)
trainer.fit(module)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/pytorch_lightning/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/pytorch_lightning/loops/optimization/automatic.py:134: `training_step` returned `None`. If this was on purpose, ignore this warning...


Stopping: reward 500.0 ≥ target 475


In [31]:

from tsilva_notebook_utils.video import render_video_from_batches
from PIL import Image

def play(module, n_episodes=1):
    env = gym.make(CONFIG['env_id'], render_mode='rgb_array')
    frames = []
    for _ in range(n_episodes):
        state, _ = env.reset(seed=CONFIG['seed'])
        done = False
        while not done:
            frame = env.render()
            pil_frame = Image.fromarray(frame)
            frames.append(pil_frame)
            state_t = torch.tensor(state, dtype=torch.float32, device=module.device).unsqueeze(0)
            with torch.no_grad():
                action = torch.argmax(module.q_model(state_t), dim=1).item()
            next_state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            state = next_state
    env.close()
    print(frames[0])
    return render_video_from_batches(frames)

play(module)

<PIL.Image.Image image mode=RGB size=600x400 at 0x7AA43D2CCF50>


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


In [ ]:
# TODO: disconnect when idle